# Autoencoders & GANs
MNIST Image Compression + Face Generation Concepts

In [ ]:
import micropip
await micropip.install(['scikit-learn','matplotlib','numpy'])
print('Ready!')

## Autoencoder Architecture

```
Input → Encoder → Latent Code (bottleneck) → Decoder → Reconstruction
  64       32→16         16 dims              16→32        64
```

**Uses:** Compression, denoising, anomaly detection, feature learning

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
digits = load_digits()
X = digits.data / 16.0
X_train,X_test = train_test_split(X,test_size=0.2,random_state=42)
print(f"Original: {X.shape[1]} dims | Bottleneck: 16 dims | Compression: 4x")

In [ ]:
ae = MLPRegressor(hidden_layer_sizes=(32,16,32),activation='relu',max_iter=500,random_state=42)
ae.fit(X_train,X_train)  # target = input
X_rec = ae.predict(X_test)
from sklearn.metrics import mean_squared_error
print(f"Reconstruction MSE: {mean_squared_error(X_test,X_rec):.4f}")
fig,axes = plt.subplots(2,10,figsize=(14,3))
for i in range(10):
    axes[0,i].imshow(X_test[i].reshape(8,8),cmap='gray'); axes[0,i].axis('off')
    axes[1,i].imshow(X_rec[i].reshape(8,8),cmap='gray');  axes[1,i].axis('off')
axes[0,0].set_ylabel('Original'); axes[1,0].set_ylabel('Reconstructed')
plt.suptitle('Autoencoder: Original vs Reconstructed'); plt.tight_layout(); plt.show()

## Denoising Autoencoder

In [ ]:
noise = 0.3
X_noisy_tr = np.clip(X_train + noise*np.random.randn(*X_train.shape),0,1)
X_noisy_te = np.clip(X_test  + noise*np.random.randn(*X_test.shape),0,1)
denoiser = MLPRegressor(hidden_layer_sizes=(32,16,32),activation='relu',max_iter=500,random_state=42)
denoiser.fit(X_noisy_tr,X_train)
X_clean = denoiser.predict(X_noisy_te)
fig,axes = plt.subplots(3,8,figsize=(14,5))
for i in range(8):
    axes[0,i].imshow(X_test[i].reshape(8,8),cmap='gray');    axes[0,i].axis('off')
    axes[1,i].imshow(X_noisy_te[i].reshape(8,8),cmap='gray');axes[1,i].axis('off')
    axes[2,i].imshow(X_clean[i].reshape(8,8),cmap='gray');   axes[2,i].axis('off')
for label,ax in zip(['Clean','Noisy','Denoised'],axes[:,0]): ax.set_ylabel(label)
plt.suptitle('Denoising Autoencoder'); plt.tight_layout(); plt.show()

## GAN — Generator + Discriminator

Two networks competing:
- **Generator (G):** Creates fake images from random noise z ~ N(0,1)
- **Discriminator (D):** Classifies real vs fake

Training objective: G tries to fool D; D tries not to be fooled.

```python
# DCGAN for face generation (PyTorch — CelebA on Kaggle)
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0),  # 4x4
            nn.BatchNorm2d(512), nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1),    # 8x8
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 3, 4, 2, 1),      # 16x16 RGB
            nn.Tanh()
        )
    def forward(self, z):
        return self.net(z.view(-1, 100, 1, 1))
```

**Dataset for face generation:** CelebA (202K celebrity faces) on Kaggle.

## VAE vs GAN

| | Autoencoder | VAE | GAN |
|---|---|---|---|
| Purpose | Compression | Smooth latent space | Realistic synthesis |
| Training | MSE loss | ELBO | Adversarial |
| Use | Anomaly detection | Drug discovery | Image/video gen |

---
**Your turn:** Increase noise to 0.5 and retrain. Can the denoiser still recover clean digits?